In [1]:
from google.colab import userdata
key = userdata.get("GEMINI_API_KEY")
print("key loaded:", bool(key))

key loaded: True


In [11]:
# Cell 1 lets you ask Gemini a question from the notebook, and it saves every answer in a file.

!pip install -q google-genai

import os, json, time, hashlib
from google import genai
from google.colab import userdata

client = genai.Client(api_key=userdata.get("GEMINI_API_KEY"))
MODEL = "gemini-3.5-flash"
CACHE_PATH = "gemini_cache.json"
cache = json.load(open(CACHE_PATH)) if os.path.exists(CACHE_PATH) else {}
last_call = [0.0]

def call_model(prompt):
    key = hashlib.sha256((MODEL + "||" + prompt).encode()).hexdigest()
    if key in cache:
        return cache[key]
    wait = 4.5 - (time.time() - last_call[0])
    if wait > 0:
        time.sleep(wait)
    resp = client.models.generate_content(model=MODEL, contents=prompt)
    last_call[0] = time.time()
    cache[key] = resp.text
    json.dump(cache, open(CACHE_PATH, "w"))
    return resp.text

print(call_model("Say hello in one short sentence."))

# OUTPUT
# print(call_model(...))  -> function: call_model(prompt)
#                            output: a short greeting written by Gemini
#                            first run: slow (real request), second run: instant (from the file)

#  prompt: "Say hello in one short sentence."
#        │
#        ▼
#  ┌────────────────────────────────────────────┐
#  │ call_model: same prompt asked before?      │
#  └───────────────┬─────────────────┬──────────┘
#                  │ yes             │ no
#                  ▼                 ▼
#     return the saved answer   ┌──────────────────────────┐
#     (instant, free)           │ wait a few seconds       │
#                               │ ask Gemini               │
#                               │ save the answer to file  │
#                               └────────────┬─────────────┘
#                                            ▼
#                                  return the answer

Hello, I hope you are having a great day!


In [16]:
# Cell 2 compares the skills in an offer with the skills in a CV, and says if the student qualifies, is almost there, is not suited, or if the offer is too vague.

!pip install -q sentence-transformers

import pandas as pd
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2") # no LLm = SentenceTransformer
TAU = 0.50 # threshold Score >= 50 Skill matches.

def match(requirements, cv_skills): # match every requirement with every skill and give a score, -1 <= score <= 1, using sims
    R = model.encode(requirements, normalize_embeddings=True)
    S = model.encode(cv_skills, normalize_embeddings=True)
    sims = R @ S.T
    out = []
    for i, req in enumerate(requirements):
        j = sims[i].argmax()
        out.append((req, cv_skills[j], float(sims[i][j])))
    return out

def decide(results, tau=TAU):
    if not results:
        return "refuse", []  # No requirements to evaluate -> refuse
    missing = [req for req, _, score in results if score < tau]
    verdict = {0: "qualifies", 1: "almost"}.get(len(missing), "not suited")
    return verdict, missing

def label(score): # text next to each score: high >= 0.75, mid >= TAU, low < TAU (low = missing)
    if score >= 0.75:
        return "high"
    if score >= TAU:
        return "mid"
    return "low"

requirements = ["Python", "Requêtes SQL", "Tableaux de bord Power BI", "Anglais professionnel"]
cv_skills = ["Programmation Python", "SQL / PostgreSQL", "Anglais courant (B2)", "Git / GitHub"]

res = match(requirements, cv_skills)
for req, skill, score in res:
    print(f"{req:30} -> {skill:25} {score:.2f}  {label(score)}")
print(decide(res))

# sims: a score for every pair (rows = requirements, columns = CV skills)
R = model.encode(requirements, normalize_embeddings=True)
S = model.encode(cv_skills, normalize_embeddings=True)
sims = R @ S.T
print("\nsims = R @ S.T")
print(pd.DataFrame(sims, index=requirements, columns=cv_skills).round(2))

# OFFER requirements                      CV skills
# ┌───────────────────────────┐           ┌───────────────────────────┐
# │ Python                    │           │ Programmation Python      │
# │ Requêtes SQL              │           │ SQL / PostgreSQL          │
# │ Tableaux de bord Power BI │           │ Anglais courant (B2)      │
# │ Anglais professionnel     │           │ Git / GitHub              │
# └─────────────┬─────────────┘           └─────────────┬─────────────┘
#               │                                       │
#               ▼                                       ▼
#        ┌─────────────────────────────────────────────────────┐
#        │ model.encode: turns every text into numbers         │
#        └──────────────────────────┬──────────────────────────┘
#                                   ▼
#        ┌─────────────────────────────────────────────────────┐
#        │ sims = R @ S.T: a score for every pair              │
#        └──────────────────────────┬──────────────────────────┘
#                                   ▼
#        ┌─────────────────────────────────────────────────────┐
#        │ match(): for each requirement, keep the closest     │
#        │ skill and its score                                 │
#        │                                                     │
#        │ Python         -> Programmation Python    high      │
#        │ Requêtes SQL   -> SQL / PostgreSQL        mid       │
#        │ Power BI       -> (nothing close)         low       │
#        │ Anglais pro    -> Anglais courant (B2)    mid       │
#        └──────────────────────────┬──────────────────────────┘
#                                   ▼
#        ┌─────────────────────────────────────────────────────┐
#        │ decide(): score below TAU (0.50) = missing          │
#        └──────────────────────────┬──────────────────────────┘
#                                   ▼
#        ┌──────────────┬───────────┴──────────┬───────────────┐
#        ▼              ▼                      ▼               ▼
#    0 missing      1 missing             2+ missing     no requirements
#    qualifies        almost              not suited         refuse
#                       ▲
#                       └── this test: only Power BI is missing

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Python                         -> Programmation Python      0.92  high
Requêtes SQL                   -> SQL / PostgreSQL          0.67  mid
Tableaux de bord Power BI      -> Anglais courant (B2)      0.19  low
Anglais professionnel          -> Anglais courant (B2)      0.59  mid
('almost', ['Tableaux de bord Power BI'])

sims = R @ S.T
                           Programmation Python  SQL / PostgreSQL  \
Python                                     0.92              0.00   
Requêtes SQL                               0.21              0.67   
Tableaux de bord Power BI                 -0.02              0.08   
Anglais professionnel                      0.25             -0.08   

                           Anglais courant (B2)  Git / GitHub  
Python                                     0.30          0.11  
Requêtes SQL                               0.04          0.00  
Tableaux de bord Power BI                  0.19          0.13  
Anglais professionnel                      0.59          0.